In [47]:
import pandas as pd
import json
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import pytz
import requests
import psycopg2

In [48]:
def read_db_credentials(path="data/config.txt"):
    creds = {}
    with open(path, "r") as f:
        for line in f:
            key, value = line.strip().split("=")
            creds[key] = value
    return creds

def connect_to_db(creds):
    return psycopg2.connect(
        host=creds["host"],
        port=creds["port"],
        dbname=creds["database"],
        user=creds["user"],
        password=creds["password"]
    )


In [49]:
with open("strava_export_combined.json") as f:
    strava_data = json.load(f)


In [50]:
# Top-Level Keys anzeigen
data = strava_data
import json

def print_json_structure(data, indent=0):
    spacer = "  " * indent
    if isinstance(data, dict):
        for key, value in data.items():
            print(f"{spacer}\"{key}\": ", end="")
            if isinstance(value, (dict, list)):
                print()
                print_json_structure(value, indent + 1)
            else:
                print(type(value).__name__)
    elif isinstance(data, list):
        print(f"{spacer}[")
        if data:
            print_json_structure(data[0], indent + 1)
        else:
            print(f"{'  ' * (indent + 1)}<empty>")
        print(f"{spacer}]")
    else:
        print(f"{spacer}{type(data).__name__}")

# JSON-Datei laden

# Struktur ausgeben
print_json_structure(data)


"flags": 
  [
    <empty>
  ]
"media": 
  [
    <empty>
  ]
"contacts": 
  [
    <empty>
  ]
"segments": 
  [
    <empty>
  ]
"privacy_zones": 
  [
    <empty>
  ]
"local_legend_segments": 
  [
    <empty>
  ]
"applications": 
  [
    <empty>
  ]
"orders": 
  [
    <empty>
  ]
"social_settings": 
  [
    "Automatisches Teilen auf Facebook ist aktiviert": str
    "Beliebte Garmin-Segmente aktiviert": str
    "Standard-Highlight-Bild für Aktivitäten": str
    "Hauptclub-ID-Nummer": float
    "Kontakte synchronisieren aktiviert": str
    "Keine Mitteilungen über meine Aktivitäten an meine Abonnenten senden": str
  ]
"bikes": 
  [
    <empty>
  ]
"starred_segments": 
  [
    <empty>
  ]
"posts": 
  [
    <empty>
  ]
"global_challenges": 
  [
    <empty>
  ]
"shoes": 
  [
    <empty>
  ]
"components": 
  [
    <empty>
  ]
"connected_apps": 
  [
    "Anwendungsname aktiviert": str
  ]
"general_preferences": 
  [
    "Geburtsdatum": str
    "Gewicht": str
    "Funktionale Schwellenleistung": 

# ACTIVITIES

In [51]:
activities = strava_data.get("activities", [])

#TODO english version


strava_activity_types = [
    "Lauf", "Traillauf", "Spaziergang", "Wandern", "Virtueller Lauf",
    "Radfahrt", "Mountainbike-Fahrt", "Gravel-Fahrt", "E-Bike-Radfahrt", "E-Mountainbike-Fahrt",
    "Velomobil", "Virtuelle Radfahrt", "Kanu", "Kajakfahrt", "Kitesurfen", "Rudern",
    "Stand-up-Paddling", "Surfen", "Schwimmen", "Windsurfen", "Eislaufen", "Ski Alpin",
    "Tourenski", "Ski Nordisch", "Snowboarden", "Schneeschuhwanderung", "Handbike-Fahrt",
    "Inlineskaten", "Klettern", "Rollski", "Golf", "Skateboarden", "Fußball", "Rollstuhlfahrt",
    "Badminton", "Tennis", "Pickleball", "Crossfit", "Crosstrainer", "Stufen-Stepper",
    "Gewichtstraining", "Yoga", "Training", "HIIT", "Pilates", "Tischtennis", "Squash",
    "Racquetball"
]

# Mapping basierend auf typischer Kategorisierung
def classify_user_activity_type(activity):
    cardio = ["lauf", "trail", "spazier", "wandern", "virtuel", "rad", "gravel", "velo", "bike",
              "kajak", "kanu", "rudern", "paddling", "surf", "schwimm", "ski", "snow", "skate", "inlineskate", "eis"]
    strength = ["crossfit", "gewicht", "hiit"]
    flexibility = ["yoga", "pilates"]
    wellness = ["golf", "spazier", "wandern", "tischtennis", "badminton", "tennis", "squash", "pickle", "racquet"]

    a = activity.lower()
    if any(k in a for k in strength):
        return "Strength"
    elif any(k in a for k in flexibility):
        return "Flexibility"
    elif any(k in a for k in wellness):
        return "Wellness"
    elif any(k in a for k in cardio):
        return "Cardio"
    else:
        return "Other"

extracted_rows = []
for act in activities:
    art = act.get("Aktivitätsart", "Unbekannt")
    dauer = act.get("Verstrichene Zeit", 0)
    distanz = act.get("Distanz.1")/1000
    timestamp = act.get("Aktivitätsdatum")
    dauer_min = round(dauer / 60, 2)
    extracted_rows.append({
        "activity": art,
        "timestamp": pd.to_datetime(timestamp, format="%d.%m.%Y, %H:%M:%S"),
        "duration": dauer_min,
        "distance": distanz
    })

df_activities = pd.DataFrame(extracted_rows)
df_activities ["type"] = df_activities ["activity"].apply(classify_user_activity_type)
print(df_activities)



    activity           timestamp  duration    distance    type
0   Radfahrt 2024-02-25 12:53:20    219.98   51.580738  Cardio
1   Radfahrt 2024-02-28 11:56:53    275.03   56.089398  Cardio
2   Radfahrt 2024-05-19 08:46:09    141.18   35.139141  Cardio
3   Radfahrt 2024-05-19 12:26:18     27.48    7.808250  Cardio
4   Radfahrt 2024-05-30 08:57:37    178.65   31.187990  Cardio
5   Radfahrt 2024-06-02 10:02:45    189.80   65.709898  Cardio
6   Radfahrt 2024-06-07 17:15:37    149.40   38.424551  Cardio
7   Radfahrt 2024-06-09 15:13:13    249.47   63.315859  Cardio
8   Radfahrt 2024-06-16 11:19:39    353.70   95.612742  Cardio
9   Radfahrt 2024-06-28 18:27:47    126.23   25.425900  Cardio
10  Radfahrt 2024-07-21 15:06:07    208.62   49.371582  Cardio
11  Radfahrt 2024-08-11 15:52:27    197.92   45.415141  Cardio
12  Radfahrt 2024-08-20 14:40:13    161.70   47.986688  Cardio
13  Radfahrt 2024-09-01 15:44:23    192.48   67.445391  Cardio
14  Radfahrt 2024-09-04 10:08:52    543.73  158.377594 

In [52]:
df_steps = pd.read_pickle("tmp/activites.pkl")
df_activites_tot = pd.concat([df_activities,df_steps])
print(df_activites_tot)

      activity                  timestamp  duration   distance      type
0     Radfahrt        2024-02-25 12:53:20    219.98  51.580738    Cardio
1     Radfahrt        2024-02-28 11:56:53    275.03  56.089398    Cardio
2     Radfahrt        2024-05-19 08:46:09    141.18  35.139141    Cardio
3     Radfahrt        2024-05-19 12:26:18     27.48   7.808250    Cardio
4     Radfahrt        2024-05-30 08:57:37    178.65  31.187990    Cardio
...        ...                        ...       ...        ...       ...
7298      Walk  2025-06-09 19:48:34+02:00     64.42   6.292000  Wellness
7361      Walk  2025-06-12 16:51:58+02:00    110.68  10.339000  Wellness
7429      Walk  2025-06-15 20:34:52+02:00    107.87  10.578000  Wellness
7458      Walk  2025-06-17 07:46:27+02:00     66.77   5.897000  Wellness
7517      Walk  2025-06-20 07:38:30+02:00    111.17   6.089000  Wellness

[135 rows x 5 columns]


In [53]:
df_activites_score = df_activites_tot
df_activites_score["score"] = 0

df_activites_score.loc[df_activites_score["type"] == "Cardio", "score"] = df_activites_score["duration"] * 0.6 + df_activites_score["distance"] * 0.4
df_activites_score.loc[df_activites_score["type"] == "Strength", "score"] = 10  # z. B. je Session
df_activites_score.loc[df_activites_score["type"] == "Wellness", "score"] = df_activites_score["duration"]
df_activites_score.loc[df_activites_score["type"] == "Flexibility", "score"] = df_activites_score["duration"]
score_summary = df_activites_score.groupby("type")["score"].sum()
# Normalisierte Anteile berechnen
score_parts = score_summary / score_summary.sum()


print(score_summary)

type
Cardio      3759.253675
Wellness    7936.770000
Name: score, dtype: float64


/var/folders/n9/9rvxcg4d1d15_3wh3wq9f0l40000gn/T/ipykernel_1871/328033346.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[152.62029531 187.45375937  98.76365625  19.6113     119.66519609
 140.16395938 105.00982031 175.00834375 250.46509688  85.90836016
 144.92063281 136.91805625 116.214675   142.46615625 389.5890375
  91.32216406 110.07084844 248.52491875 150.51944063  85.41315938
   1.752       84.36144063  25.79716016  24.46036016 123.17275938
  17.02431992 118.15755937 215.73991875  98.33675937  99.82252031]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_activites_score.loc[df_activites_score["type"] == "Cardio", "score"] = df_activites_score["duration"] * 0.6 + df_activites_score["distance"] * 0.4


# MOTIVATION

In [54]:
def get_motivation(data):
    score = {
        "Leistung": 0,
        "Soziale Anerkennung": 0,
        "Gamification/Wettbewerb": 0
    }

    # Ziele oder leistungsmotivierte Aktivitätsmetriken
    if len(data.get("goals", [])) > 0:
        score["Leistung"] += 2

    for a in data.get("activities", []):
        if not isinstance(a, dict):
            continue
        if a.get("Relative Leistung") is not None:
            score["Leistung"] += 1
        if a.get("Gefühlte Anstrengung") is not None:
            score["Leistung"] += 1

    # Soziales Verhalten
    social_settings = data.get("social_settings", [{}])
    if social_settings and isinstance(social_settings[0], dict):
        for v in social_settings[0].values():
            if isinstance(v, str) and "aktiviert" in v.lower():
                score["Soziale Anerkennung"] += 1

    # Kudos aus reactions
    kudo_count = sum(
        1 for r in data.get("reactions", [])
        if isinstance(r, dict) and r.get("Reaktionstyp", "").lower() == "kudos"
    )
    score["Soziale Anerkennung"] += kudo_count * 1

    # Herausforderungen
    if len(data.get("global_challenges", [])) > 0:
        score["Gamification/Wettbewerb"] += 2
    if len(data.get("group_challenges", [])) > 0:
        score["Gamification/Wettbewerb"] += 2

    # Bewertung: stärkste Motivation
    if all(v == 0 for v in score.values()):
        return "Unklar"
    
    # Max-Wert und Rückgabe der zugehörigen Motivation
    dominant = max(score, key=score.get)
    return {"dominant": dominant, **score}


print(get_motivation(strava_data))

{'dominant': 'Leistung', 'Leistung': 60, 'Soziale Anerkennung': 56, 'Gamification/Wettbewerb': 0}


# EVENT DOG

# STRAVA SOCIAL SCORE

In [55]:
from datetime import datetime

def get_social_interaction_score(data):
    followers = len(data.get("followers", []))
    following = len(data.get("following", []))
    clubs = len(data.get("clubs", [])) + len(data.get("memberships", []))

    # Zeitspanne bestimmen (von erster bis letzter Aktivität)
    dates = [
        datetime.strptime(a["Aktivitätsdatum"], "%d.%m.%Y, %H:%M:%S")
        for a in data.get("activities", [])
        if isinstance(a, dict) and "Aktivitätsdatum" in a
    ]

    if not dates:
        active_weeks = 1  # Annahme, falls keine Daten
    else:
        delta_days = (max(dates) - min(dates)).days
        active_weeks = max(delta_days // 7, 1)

    # Pro-Woche-Berechnung
    comments_total = len(data.get("comments", []))
    reactions_total = len(data.get("reactions", []))
    comments_per_week = comments_total / active_weeks
    reactions_per_week = reactions_total / active_weeks

    # Normierung: z. B. 5 pro Woche = Maximum (1.0)
    norm_comments = min(comments_per_week / 1, 1)
    norm_reactions = min(reactions_per_week / 3, 1)

    # Feste Grenzwerte für andere Metriken
    score = (
        min(followers / 20, 1) +
        min(following / 20, 1) +
        min(clubs / 5, 1) +
        norm_comments +
        norm_reactions
    ) / 5

    return round(score, 2)



In [56]:

def parse_activity_date(date_str):
    try:
        return datetime.strptime(date_str, "%d.%m.%Y, %H:%M:%S")
    except Exception:
        return None

def attends_events(data):
    keywords = ["rennen", "marathon", "event", "race", "triathlon", "5k", "10k"]

    # Events zählen
    event_dates = []
    for e in data.get("events", []):
        date = e.get("Datum") or e.get("date")  # je nach Export
        dt = parse_activity_date(date) if date else None
        if dt:
            event_dates.append(dt.year)

    # Aktivitäten nach Name durchsuchen
    for a in data.get("activities", []):
        if not isinstance(a, dict):
            continue
        name = a.get("Name der Aktivität", "").lower()
        if any(kw in name for kw in keywords):
            dt = parse_activity_date(a.get("Aktivitätsdatum", ""))
            if dt:
                event_dates.append(dt.year)

    # Jahresweise zählen
    from collections import Counter
    year_counts = Counter(event_dates)

    # Prüfen, ob in irgendeinem Jahr mehr als 2 vorkommen
    return any(count > 2 for count in year_counts.values())


In [57]:
activity_counts = df_activites_tot["activity"].value_counts().head(4).reset_index()
activity_counts.columns = ["activity", "count"]
df_activites_tot["timestamp"] = pd.to_datetime(df_activites_tot["timestamp"], errors="coerce")

def safe_activity_label(df, idx):
    if idx < len(df):
        row = df.iloc[idx]
        return f"{row['activity']}\n{row['count']}"
    else:
        return "NA"


In [60]:
import numpy as np
user_number = 1
strength_pw_mean = (
    df_activites_tot[df_activites_tot["type"] == "Strength"]
    .groupby(df_activites_tot["timestamp"].dt.to_period("W"))
    .size()
    .mean()
)

# Falls Ergebnis NaN ist, ersetze durch None
fl_workouts_pw = 0 if np.isnan(strength_pw_mean) else float(strength_pw_mean)
fl_cardio_min_pw = (
    df_activites_tot[df_activites_tot["type"] == "Cardio"]
    .groupby(df_activites_tot["timestamp"].dt.to_period("W"))["duration"]
    .sum()
    .mean()
)
pa_t1 = safe_activity_label(activity_counts, 0)
pa_t2 = safe_activity_label(activity_counts, 1)
pa_t3 = safe_activity_label(activity_counts, 2)
pa_t4 = safe_activity_label(activity_counts, 3)
fintess_type_endurence = score_parts.get("Cardio", 0)
fitness_type_strength = score_parts.get("Strength", 0)
fintess_type_flexibility = score_parts.get("fleibilty", 0)
fintess_type_relaation = score_parts.get("Wellness", 0)
motivation_type = get_motivation(strava_data)["dominant"]
sc_strava_social_score = get_social_interaction_score(strava_data)
sc_strava_event_dog= attends_events(strava_data)



In [61]:

creds = read_db_credentials()
conn = connect_to_db(creds)
cur = conn.cursor()

sql = """
INSERT INTO dim_strava (
    user_number,
    fl_workouts_pw,
    fl_cardio_min_pw,
    pa_t1,
    pa_t2,
    pa_t3,
    pa_t4,
    fitness_type_endurence,
    fitness_type_strength,
    fitness_type_flexibility,
    fitness_type_relaation,
    motivation_type,
    sc_strava_social_score,
    sc_strava_event_dog
) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

values = (
    int(user_number),
    float(fl_workouts_pw),
    float(fl_cardio_min_pw),
    str(pa_t1),
    str(pa_t2),
    str(pa_t3),
    str(pa_t4),
    float(fintess_type_endurence),
    float(fitness_type_strength),
    float(fintess_type_flexibility),
    float(fintess_type_relaation),
    str(motivation_type),
    float(sc_strava_social_score),
    bool(sc_strava_event_dog)
)

cur.execute(sql, values)
conn.commit()
cur.close()
conn.close()

print("✅ Daten erfolgreich in dim_strava gespeichert.")


✅ Daten erfolgreich in dim_strava gespeichert.
